# Summary_Day17_online.ipynb  
## 자연어처리 NLP 종합실습 · BiLSTM · DistilBERT · Tokenizer · Transformer

이번 17강은 **기존 코드 재활용 + 자연어처리 NLP 종합 실습** 흐름으로 정리한다.

앞 강의까지는 이미지 데이터를 중심으로 CNN, pretrained model, custom dataset을 다뤘다.  
17강에서는 데이터 형태가 이미지에서 **텍스트**로 바뀐다.

강사님이 강조한 큰 흐름은 다음이다.

```text
이미지 분류: image, label
자연어 분류: text, label
```

즉, 문제 구조는 비슷하다.  
하지만 텍스트는 바로 Tensor가 아니므로 모델에 넣기 전에 반드시 숫자로 바꾸어야 한다.

전체 흐름은 다음이다.

```text
자연어처리 NLP 큰 그림
→ 텍스트 전처리
→ 토큰화 Tokenization
→ 어휘사전 Vocab
→ 인덱스 변환 Numericalization
→ 패딩 Padding
→ 임베딩 Embedding
→ RNN/LSTM/BiLSTM
→ IMDb 감성 분류
→ DistilBERT 파인튜닝
→ Trainer 사용
→ 평가 지표 accuracy, precision, recall, f1
→ Attention, Transformer, BERT, GPT 개념 정리
→ 과제 빈칸 패턴 정리
```

이 파일은 **인터넷 가능 버전**이다.  
Hugging Face `datasets`, `transformers`, IMDb dataset, DistilBERT 모델 다운로드가 가능하다는 전제로 작성했다.

> 필기 포인트:  
> 자연어처리에서 가장 먼저 해야 할 일은 문장을 숫자 sequence로 바꾸는 것이다.  
> 텍스트 모델은 글자를 그대로 이해하는 것이 아니라, token id와 embedding을 통해 계산한다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. 자연어처리에서 텍스트를 모델 입력으로 바꾸는 과정을 이해한다.
2. 정규식 기반 간단 tokenizer를 직접 만든다.
3. IMDb 감성 분류 데이터셋을 불러온다.
4. `Counter`로 어휘사전을 만들고 `stoi`, `itos` 구조를 이해한다.
5. 문장을 token id로 바꾸고 padding/truncation을 적용한다.
6. PyTorch Dataset과 DataLoader로 텍스트 batch를 만든다.
7. `Embedding → BiLSTM → Linear` 구조로 감성 분류 모델을 만든다.
8. DistilBERT tokenizer와 model을 불러와 fine-tuning 흐름을 이해한다.
9. `DataCollatorWithPadding`, `TrainingArguments`, `Trainer`를 정리한다.
10. Attention, Transformer, BERT, GPT의 큰 차이를 시험용으로 정리한다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import re
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
```

- `re`: 정규식으로 토큰을 추출할 때 사용한다.
- `Counter`: 단어 빈도수를 셀 때 사용한다.
- `load_dataset`: Hugging Face 데이터셋을 불러온다.
- `AutoTokenizer`: pretrained model에 맞는 tokenizer를 자동으로 불러온다.
- `AutoModelForSequenceClassification`: 문장 분류용 pretrained model을 불러온다.
- `DataCollatorWithPadding`: batch마다 가장 긴 문장에 맞춰 동적 padding을 적용한다.
- `Trainer`: Hugging Face 방식의 학습 루프를 대신 실행해 준다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

try:
    from datasets import load_dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
    import evaluate
    HF_AVAILABLE = True
except Exception as e:
    HF_AVAILABLE = False
    print("Hugging Face 관련 라이브러리가 없거나 로드되지 않았다.")
    print("Colab에서는 다음 설치가 필요하다.")
    print('!pip install -q "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2"')
    print("error:", e)

torch.manual_seed(2025)
np.random.seed(2025)
random.seed(2025)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)
print("HF_AVAILABLE:", HF_AVAILABLE)

## 3. device 설정과 seed 고정

### 함수 사용법

```python
device = "cuda" if torch.cuda.is_available() else "cpu"
```

- GPU가 있으면 `cuda`를 사용한다.
- GPU가 없으면 `cpu`를 사용한다.
- model과 Tensor는 같은 device에 있어야 한다.

In [ ]:
SEED = 2025

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)

## 4. NLP 전체 그림

자연어처리는 인간의 언어를 컴퓨터가 계산할 수 있는 숫자 형태로 바꾸고, 그 숫자 sequence에서 의미를 학습하는 분야다.

강의에서 강조한 점은 다음이다.

```text
자연어는 전처리가 어렵다.
사람은 문법대로만 말하지 않는다.
비속어, 오타, 줄임말, 문맥, 어순이 모두 영향을 준다.
```

이미지 분류와 비교하면 다음과 같다.

| 이미지 분류 | 자연어 분류 |
|---|---|
| image tensor | text |
| label | label |
| transform | tokenization / padding |
| CNN | RNN, LSTM, Transformer |
| pixel pattern | word sequence / context |

> 강의식 표현으로 정리하면, 자연어는 “원본이 깨끗해야 한다”는 전처리 문제가 크다.

## 5. 토큰화 Tokenization

토큰화는 문장을 작은 단위로 자르는 작업이다.

예를 들어 다음 문장을 생각한다.

```text
I really loved this movie!
```

토큰화 결과는 대략 다음과 같다.

```text
["i", "really", "loved", "this", "movie"]
```

### 함수 사용법

```python
token_pattern = re.compile(r"[a-z0-9']+")
token_pattern.findall(text.lower())
```

- `text.lower()`: 대문자를 소문자로 바꾼다.
- `findall`: 정규식에 맞는 모든 token을 찾는다.
- `r"[a-z0-9']+"`: 영어 소문자, 숫자, apostrophe를 token으로 잡는다.

In [ ]:
token_pattern = re.compile(r"[a-z0-9']+")

def simple_tokenize(text: str):
    return token_pattern.findall(text.lower())

sample_text = "I really loved this movie! It wasn't boring."

tokens = simple_tokenize(sample_text)

print("원문:", sample_text)
print("토큰:", tokens)

코드 한 줄씩 의미:

```python
token_pattern = re.compile(...)
```

정규식 패턴을 미리 준비한다.

```python
text.lower()
```

대소문자 차이를 줄이기 위해 소문자로 바꾼다.

```python
findall(...)
```

문장에서 조건에 맞는 token을 전부 찾는다.

> 강의 포인트:  
> 토큰은 API에서 말하는 token과 같은 큰 개념이다.  
> 문장을 모델이 처리할 수 있는 작은 조각으로 나누는 단위다.

## 6. IMDb 데이터셋 로드

IMDb는 영화 리뷰 문장을 긍정/부정으로 분류하는 대표 감성 분석 데이터셋이다.

### 함수 사용법

```python
raw = load_dataset("stanfordnlp/imdb")
```

- `raw["train"]`: 학습 데이터다.
- `raw["test"]`: 테스트 데이터다.
- 각 샘플은 보통 `text`, `label`을 가진다.
- label은 `0=negative`, `1=positive`다.

In [ ]:
if HF_AVAILABLE:
    raw = load_dataset("stanfordnlp/imdb")
    print(raw)

    print("첫 샘플 text 일부:")
    print(raw["train"][0]["text"][:300])
    print("첫 샘플 label:", raw["train"][0]["label"])
else:
    raw = None
    print("Hugging Face datasets를 사용할 수 없어 이 셀은 건너뛴다.")

## 7. 어휘사전 Vocab 만들기

컴퓨터는 token 문자열을 그대로 계산하지 못한다.  
그래서 token을 숫자 id로 바꾸는 어휘사전이 필요하다.

강의 코드의 핵심 변수는 다음이다.

```text
counter: token 빈도수 저장
itos: index to string
stoi: string to index
PAD: padding token
UNK: unknown token
```

### 함수 사용법

```python
Counter()
counter.update(tokens)
counter.most_common(n)
```

- `Counter`: 빈도수를 세는 딕셔너리형 도구다.
- `update`: token list를 넣어 빈도를 누적한다.
- `most_common`: 많이 등장한 token 순서대로 가져온다.

In [ ]:
MAX_VOCAB = 30000
PAD, UNK = "<pad>", "<unk>"

if HF_AVAILABLE and raw is not None:
    counter = Counter()

    # 빠른 실행을 위해 앞부분 일부로 vocab을 만든다.
    # 전체 원본처럼 하려면 raw["train"] 전체를 순회하면 된다.
    for ex in raw["train"].select(range(5000)):
        counter.update(simple_tokenize(ex["text"]))

    most_common = counter.most_common(MAX_VOCAB - 2)

    itos = [PAD, UNK] + [token for token, count in most_common]
    stoi = {token: index for index, token in enumerate(itos)}

    PAD_IDX = stoi[PAD]
    UNK_IDX = stoi[UNK]

    print("어휘 크기:", len(itos))
    print("상위 10개:", most_common[:10])
    print("itos[:10]:", itos[:10])
    print("stoi['the']:", stoi.get("the"))
else:
    counter = None
    itos = [PAD, UNK]
    stoi = {PAD: 0, UNK: 1}
    PAD_IDX, UNK_IDX = 0, 1
    print("데모용 vocab만 생성했다.")

## 8. 문장을 index sequence로 바꾸기

문장을 모델에 넣으려면 다음 순서를 거친다.

```text
문장
→ token list
→ token id list
→ max length 기준 자르기
→ 부족하면 PAD로 채우기
```

### 함수 사용법

```python
ids = [stoi.get(tok, UNK_IDX) for tok in tokens]
ids = ids[:MAX_LEN]
ids += [PAD_IDX] * (MAX_LEN - len(ids))
```

- `stoi.get(tok, UNK_IDX)`: 사전에 있으면 id, 없으면 `<unk>` id를 사용한다.
- `ids[:MAX_LEN]`: 긴 문장은 자른다.
- padding: 짧은 문장은 `<pad>` id로 길이를 맞춘다.

In [ ]:
MAX_LEN = 256

def encode(text: str):
    tokens = simple_tokenize(text)
    ids = [stoi.get(tok, UNK_IDX) for tok in tokens][:MAX_LEN]

    if len(ids) < MAX_LEN:
        ids += [PAD_IDX] * (MAX_LEN - len(ids))

    return ids

def encode_label(y: int):
    return int(y)

encoded_sample = encode(sample_text)

print("encoded length:", len(encoded_sample))
print("encoded 앞 20개:", encoded_sample[:20])
print("PAD_IDX:", PAD_IDX)
print("UNK_IDX:", UNK_IDX)

## 9. PyTorch Dataset 만들기

Hugging Face dataset을 바로 PyTorch DataLoader에 넣기보다, `__getitem__`에서 text를 id Tensor로 바꿔 주는 Dataset을 만든다.

### 클래스 사용법

```python
class IMDBTensor(torch.utils.data.Dataset):
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ...
        return x, y
```

- `__len__`: 데이터 개수를 반환한다.
- `__getitem__`: index 하나에 해당하는 입력과 label을 반환한다.

In [ ]:
class IMDBTensor(torch.utils.data.Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        x = torch.tensor(encode(text), dtype=torch.long)
        y = torch.tensor(encode_label(label), dtype=torch.long)

        return x, y

if HF_AVAILABLE and raw is not None:
    # 빠른 실습용 subset이다.
    train_split = raw["train"].select(range(4000))
    test_split = raw["test"].select(range(1000))

    train_ds = IMDBTensor(train_split)
    test_ds = IMDBTensor(test_split)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=0)

    X_batch, y_batch = next(iter(train_loader))

    print("X_batch shape:", X_batch.shape)
    print("y_batch shape:", y_batch.shape)
else:
    train_loader = None
    test_loader = None
    print("Hugging Face dataset을 사용할 수 없어 Dataset 생성은 건너뛴다.")

## 10. Embedding Layer 이해

Embedding은 token id를 dense vector로 바꾸는 layer다.

### 함수 사용법

```python
nn.Embedding(vocab_size, emb, padding_idx=pad_idx)
```

- `vocab_size`: 단어 사전 크기다.
- `emb`: 각 token을 몇 차원 벡터로 표현할지 정한다.
- `padding_idx`: padding token의 embedding을 특별히 처리한다.

입력과 출력 shape은 다음이다.

```text
입력 x: [batch, seq_len]
출력 e: [batch, seq_len, embedding_dim]
```

In [ ]:
vocab_size = max(len(itos), 10)
embedding_dim = 8

embedding_layer = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)

dummy_ids = torch.tensor([[2, 3, 4, 0, 0]], dtype=torch.long)
dummy_emb = embedding_layer(dummy_ids)

print("dummy_ids shape:", dummy_ids.shape)
print("dummy_emb shape:", dummy_emb.shape)

## 11. BiLSTM 모델 정의

강의 초급 실습의 핵심 모델은 `Embedding + BiLSTM + FC`다.

```text
token ids
→ Embedding
→ BiLSTM
→ forward 마지막 hidden + backward 마지막 hidden 연결
→ Dropout
→ Linear
→ logits
```

### 함수 사용법

```python
nn.LSTM(
    input_size=emb,
    hidden_size=hidden,
    batch_first=True,
    bidirectional=True
)
```

- `input_size`: embedding 차원이다.
- `hidden_size`: LSTM hidden state 크기다.
- `batch_first=True`: 입력 shape을 `[batch, seq_len, feature]`로 쓴다.
- `bidirectional=True`: 앞에서 뒤, 뒤에서 앞 두 방향을 함께 학습한다.

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, emb=128, hidden=128, num_layers=1, num_classes=2, pad_idx=0, dropout=0.2):
        super().__init__()

        self.emb = nn.Embedding(vocab_size, emb, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            input_size=emb,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)

                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        e = self.emb(x)

        out, (h, c) = self.lstm(e)

        last_f = h[-2]
        last_b = h[-1]

        h_cat = torch.cat([last_f, last_b], dim=1)
        h_cat = self.dropout(h_cat)

        logits = self.fc(h_cat)

        return logits

model_bilstm = BiLSTM(len(itos), emb=128, hidden=128, num_layers=1, pad_idx=PAD_IDX).to(device)

dummy_x = torch.randint(0, len(itos), (4, 20)).to(device)

with torch.no_grad():
    dummy_logits = model_bilstm(dummy_x)

print(model_bilstm)
print("dummy input shape:", dummy_x.shape)
print("dummy logits shape:", dummy_logits.shape)

코드 한 줄씩 의미:

```python
self.emb = nn.Embedding(...)
```

token id를 embedding vector로 바꾼다.

```python
self.lstm = nn.LSTM(..., bidirectional=True)
```

문장을 앞 방향과 뒤 방향으로 모두 읽는다.

```python
out, (h, c) = self.lstm(e)
```

LSTM의 전체 출력과 마지막 hidden/cell state를 받는다.

```python
last_f = h[-2]
last_b = h[-1]
```

마지막 layer의 순방향 hidden과 역방향 hidden을 꺼낸다.

```python
torch.cat([last_f, last_b], dim=1)
```

두 방향의 hidden state를 feature 차원으로 연결한다.

```python
self.fc(h_cat)
```

긍정/부정 class logits를 만든다.

## 12. BiLSTM 학습/평가 함수

텍스트 분류도 이미지 분류와 학습 구조는 거의 같다.

```text
optimizer.zero_grad()
logits = model(X)
loss = criterion(logits, y)
loss.backward()
optimizer.step()
```

차이는 입력이 image Tensor가 아니라 token id Tensor라는 점이다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model_bilstm.parameters(), lr=1e-3)

def train_one_epoch_bilstm(epoch):
    model_bilstm.train()

    total_loss = 0.0
    total_correct = 0
    total = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model_bilstm(X)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    print(f"[Train] Epoch {epoch} | loss={total_loss/total:.4f} | acc={total_correct/total:.4f}")
    return total_loss / total, total_correct / total


@torch.no_grad()
def evaluate_bilstm():
    model_bilstm.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0

    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model_bilstm(X)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    print(f"[Test ] loss={total_loss/total:.4f} | acc={total_correct/total:.4f}")
    return total_loss / total, total_correct / total

## 13. BiLSTM 짧은 학습 실행

원본 코드는 2~3 epoch를 권장한다.  
여기서는 실행 시간을 줄이기 위해 기본 1 epoch만 실행한다.

In [ ]:
RUN_BILSTM_TRAINING = HF_AVAILABLE and train_loader is not None

bilstm_history = []

if RUN_BILSTM_TRAINING:
    EPOCHS = 1

    for ep in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch_bilstm(ep)
        test_loss, test_acc = evaluate_bilstm()

        bilstm_history.append([ep, train_loss, train_acc, test_loss, test_acc])

    bilstm_history = np.array(bilstm_history)

    plt.plot(bilstm_history[:, 0], bilstm_history[:, 1], label="train loss")
    plt.plot(bilstm_history[:, 0], bilstm_history[:, 3], label="test loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("BiLSTM IMDb Loss")
    plt.legend()
    plt.show()

    plt.plot(bilstm_history[:, 0], bilstm_history[:, 2], label="train acc")
    plt.plot(bilstm_history[:, 0], bilstm_history[:, 4], label="test acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title("BiLSTM IMDb Accuracy")
    plt.legend()
    plt.show()
else:
    print("BiLSTM 학습은 Hugging Face dataset이 준비되지 않아 건너뛴다.")

## 14. 과제형 BasicLSTM 빈칸 패턴

과제 파일에는 단방향 LSTM 빈칸 채우기가 나온다.

핵심 정답 패턴은 다음이다.

```python
self.embedding = nn.Embedding(vocab_size, embed_size)

self.lstm = nn.LSTM(
    input_size=embed_size,
    hidden_size=hidden_size,
    batch_first=True,
    bidirectional=False
)

self.fc = nn.Linear(hidden_size, num_classes)

lstm_out, (hn, cn) = self.lstm(emb)
logits = self.fc(hn[0])
```

단방향 LSTM은 마지막 hidden state가 `[1, batch, hidden_size]` 형태다.  
그래서 `hn[0]`을 꺼내 `fc`에 넣는다.

In [ ]:
class BasicLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)

        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=False
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)

        lstm_out, (hn, cn) = self.lstm(emb)

        logits = self.fc(hn[0])

        return logits

VOCAB_SIZE = 1000
EMBED_SIZE = 32
HIDDEN_SIZE = 64
NUM_CLASSES = 2

basic_model = BasicLSTM(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_CLASSES)

dummy_input = torch.randint(0, VOCAB_SIZE, (4, 10))
output = basic_model(dummy_input)

print("입력 shape:", dummy_input.shape)
print("출력 shape:", output.shape)

## 15. BiLSTM hidden 연결 빈칸 패턴

BiLSTM에서는 방향이 2개라 hidden state도 방향별로 나온다.

```text
hn shape: [num_layers * num_directions, batch, hidden_size]
```

num_layers=1이고 bidirectional=True이면 다음과 같다.

```text
hn[0] = forward 방향 마지막 hidden
hn[1] = backward 방향 마지막 hidden
```

연결 코드는 다음이다.

```python
hn_fwd = hn[0]
hn_bwd = hn[1]
hidden = torch.cat((hn_fwd, hn_bwd), dim=1)
```

In [ ]:
class BiLSTM_Forward(nn.Module):
    def __init__(self, hidden_size, num_classes):
        super().__init__()

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, hn):
        hn_fwd = hn[0]
        hn_bwd = hn[1]

        hidden = torch.cat((hn_fwd, hn_bwd), dim=1)

        logits = self.fc(hidden)

        return logits, hidden

HIDDEN_SIZE = 64

bilstm_forward = BiLSTM_Forward(HIDDEN_SIZE, 2)
dummy_hn = torch.randn(2, 4, HIDDEN_SIZE)

logits, concatenated_hidden = bilstm_forward(dummy_hn)

print("입력 hn shape:", dummy_hn.shape)
print("연결된 hidden shape:", concatenated_hidden.shape)
print("최종 logits shape:", logits.shape)

## 16. DistilBERT 개념

DistilBERT는 BERT를 경량화한 모델이다.

강의에서 설명한 핵심은 다음이다.

```text
BERT는 성능이 좋지만 크다.
DistilBERT는 핵심 기능을 유지하면서 더 작고 빠르게 만든 버전이다.
```

BERT 계열 모델의 장점은 Attention 기반으로 문맥을 잘 본다는 점이다.  
BiLSTM도 양방향 문맥을 보지만, 긴 문맥 이해와 사전학습 활용 측면에서는 BERT 계열이 강하다.

> 강의식 비유:  
> DistilBERT는 핵심 부품은 유지하면서 더 가볍게 만든 보급형 모델처럼 이해하면 된다.

## 17. DistilBERT Tokenizer와 Model 불러오기

### 함수 사용법

```python
AutoTokenizer.from_pretrained("distilbert-base-uncased")
AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
```

- `AutoTokenizer`: 모델 이름에 맞는 tokenizer를 자동으로 불러온다.
- `AutoModelForSequenceClassification`: 문장 분류용 head가 붙은 모델을 불러온다.
- `num_labels=2`: 부정/긍정 2 class 분류다.

In [ ]:
RUN_DISTILBERT_LOAD = HF_AVAILABLE

if RUN_DISTILBERT_LOAD:
    model_name = "distilbert-base-uncased"

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    distil_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    print("tokenizer:", tokenizer.__class__.__name__)
    print("model:", distil_model.__class__.__name__)
else:
    tokenizer = None
    distil_model = None
    print("transformers를 사용할 수 없어 DistilBERT 로드는 건너뛴다.")

## 18. DistilBERT 전처리 함수

Transformer 모델은 직접 vocab을 만들지 않는다.  
pretrained tokenizer가 tokenization과 id 변환을 처리한다.

### 함수 사용법

```python
tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)
```

- `truncation=True`: 너무 긴 문장을 자른다.
- `max_length`: 최대 token 길이다.
- padding은 여기서 하지 않고 collator가 batch 단위로 동적으로 처리하게 둘 수 있다.

In [ ]:
MAX_LEN = 256

if RUN_DISTILBERT_LOAD and raw is not None:
    def preprocess(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

    small_dataset = {
        "train": raw["train"].select(range(1000)),
        "test": raw["test"].select(range(300))
    }

    encoded_train = small_dataset["train"].map(preprocess, batched=True, remove_columns=["text"])
    encoded_test = small_dataset["test"].map(preprocess, batched=True, remove_columns=["text"])

    print(encoded_train)
    print(encoded_train[0].keys())
else:
    encoded_train = None
    encoded_test = None
    print("DistilBERT 전처리는 건너뛴다.")

## 19. DataCollatorWithPadding

문장 길이는 제각각이다.  
모든 문장을 처음부터 같은 길이로 padding하면 낭비가 생길 수 있다.

`DataCollatorWithPadding`은 batch 안에서 가장 긴 문장에 맞춰 padding한다.

### 함수 사용법

```python
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
```

- batch마다 동적으로 padding한다.
- GPU 메모리를 아낄 수 있다.
- tokenizer의 `pad_token_id`를 사용한다.

In [ ]:
if RUN_DISTILBERT_LOAD:
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    sentences = ["This is the first sentence.", "This one is shorter."]
    tokenized_inputs = tokenizer(sentences, padding=False, truncation=True)

    batch = data_collator([
        {"input_ids": ids} for ids in tokenized_inputs["input_ids"]
    ])

    print("패딩 전 input_ids:")
    print(tokenized_inputs["input_ids"])

    print("패딩 후 input_ids:")
    print(batch["input_ids"])
    print("pad_token_id:", tokenizer.pad_token_id)
else:
    data_collator = None
    print("DataCollator 예시는 건너뛴다.")

## 20. 평가 지표 compute_metrics

DistilBERT 실습에서는 accuracy, precision, recall, f1을 사용한다.

### 함수 사용법

```python
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    ...
```

- `logits`: 모델의 class 점수다.
- `labels`: 정답 label이다.
- `argmax(axis=1)`: 가장 높은 점수의 class를 예측으로 선택한다.

In [ ]:
if HF_AVAILABLE:
    acc_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    prec_metric = evaluate.load("precision")
    rec_metric = evaluate.load("recall")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = logits.argmax(axis=1)

        return {
            "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
            "precision": prec_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
            "recall": rec_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
            "f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
        }

    print("compute_metrics 준비 완료")
else:
    print("evaluate 라이브러리가 없어 compute_metrics 준비를 건너뛴다.")

## 21. TrainingArguments 호환 생성 함수

강의 코드에는 Transformers 버전별 인자 이름 차이를 감지하는 함수가 들어 있다.

버전에 따라 다음 인자가 다를 수 있다.

```text
evaluation_strategy vs eval_strategy
```

그래서 `signature(TrainingArguments.__init__)`로 지원되는 인자만 골라 넣는다.

> 실습 포인트:  
> 라이브러리 버전이 바뀌면 같은 코드도 에러가 날 수 있다.  
> 강의 코드는 이런 버전 차이를 방어하는 방식까지 보여준다.

In [ ]:
from inspect import signature

def make_training_args(**base):
    sig = signature(TrainingArguments.__init__).parameters

    defaults = dict(
        output_dir="/mnt/data/imdb_distilbert_day17",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        seed=2025,
        report_to="none",
        fp16=torch.cuda.is_available(),
        disable_tqdm=True
    )

    defaults.update(base or {})

    args = {}

    for k, v in defaults.items():
        if k in sig:
            args[k] = v

    if "evaluation_strategy" in sig:
        args["evaluation_strategy"] = defaults.get("evaluation_strategy", "epoch")
    elif "eval_strategy" in sig:
        args["eval_strategy"] = defaults.get("evaluation_strategy", "epoch")

    if "save_strategy" in sig:
        args["save_strategy"] = defaults.get("save_strategy", "no")

    return TrainingArguments(**args)

if HF_AVAILABLE:
    training_args = make_training_args()
    print(training_args)
else:
    training_args = None
    print("TrainingArguments 생성은 건너뛴다.")

## 22. Trainer 구성과 학습

Trainer는 Hugging Face 방식의 학습 루프를 대신 처리한다.

### 함수 사용법

```python
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
```

- `model`: 학습할 Transformer 모델이다.
- `args`: 학습 설정이다.
- `train_dataset`: 학습 데이터다.
- `eval_dataset`: 평가 데이터다.
- `data_collator`: batch padding 처리기다.
- `compute_metrics`: 평가 지표 함수다.

In [ ]:
RUN_DISTILBERT_TRAINING = False

if RUN_DISTILBERT_TRAINING and RUN_DISTILBERT_LOAD and encoded_train is not None:
    trainer = Trainer(
        model=distil_model,
        args=training_args,
        train_dataset=encoded_train,
        eval_dataset=encoded_test,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=tokenizer
    )

    trainer.train()

    eval_res = trainer.evaluate()
    print("평가 결과:", eval_res)
else:
    trainer = None
    print("DistilBERT 학습은 기본 실행에서 건너뛴다.")
    print("실행하려면 RUN_DISTILBERT_TRAINING = True로 바꾼다.")

## 23. DistilBERT 추론

학습된 모델 또는 pretrained 분류 모델에 문장을 넣어 긍정/부정 확률을 볼 수 있다.

### 함수 사용법

```python
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
probs = torch.softmax(model(**inputs).logits, dim=-1)
```

- `return_tensors="pt"`: PyTorch Tensor로 반환한다.
- `padding=True`: batch 길이를 맞춘다.
- `softmax`: logits를 확률처럼 해석할 수 있게 바꾼다.

In [ ]:
if RUN_DISTILBERT_LOAD:
    texts = [
        "This movie was absolutely wonderful and touching!",
        "I really hated this film. The acting was terrible."
    ]

    active_model = distil_model

    active_model.eval()

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    ).to(active_model.device)

    with torch.no_grad():
        probs = torch.softmax(active_model(**inputs).logits, dim=-1).cpu().numpy()

    for text, prob in zip(texts, probs):
        print(f"문장: {text}")
        print(f"  Negative={prob[0]:.3f}, Positive={prob[1]:.3f}")
else:
    print("DistilBERT 추론은 건너뛴다.")

## 24. Attention, Transformer, BERT, GPT 큰 그림

17강은 하루 안에 NLP를 모두 깊게 배우는 수업이 아니라, 큰 그림을 잡는 수업이다.

강의와 보강 PDF 기준 핵심은 다음이다.

| 모델/개념 | 핵심 |
|---|---|
| RNN | 순서대로 읽는 순환 신경망 |
| LSTM | 장기 의존성 문제를 완화한 RNN |
| BiLSTM | 문장을 앞뒤 양방향으로 읽는 LSTM |
| Attention | 중요한 token에 더 집중하는 구조 |
| Transformer | Attention을 중심으로 만든 구조 |
| BERT | Transformer Encoder 기반, 문맥 이해에 강함 |
| GPT | Transformer Decoder 기반, 다음 token 생성에 강함 |
| DistilBERT | BERT를 경량화한 모델 |

강사님이 강조한 자연어처리의 감각은 다음이다.

```text
언어는 순서가 있다.
한국말은 끝까지 들어봐야 한다.
다음 말은 확률적으로 예측된다.
모르면 그럴듯한 말을 할 수 있고, 이것이 hallucination과 연결된다.
```

## 25. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `NLP` | Natural Language Processing | 자연어처리 |
| `text` | 문장 데이터 | 모델 입력의 원본 |
| `label` | 정답 | 감성 분류에서는 0/1 |
| `token` | 문장을 나눈 작은 단위 | 단어, subword 등 |
| `tokenizer` | 문장을 token/id로 바꾸는 도구 | `AutoTokenizer` |
| `vocab` | 어휘사전 | token과 id 매핑 |
| `stoi` | string to index | token → id |
| `itos` | index to string | id → token |
| `PAD` | padding token | 길이 맞추기 |
| `UNK` | unknown token | 모르는 단어 |
| `MAX_LEN` | 최대 길이 | truncation/padding 기준 |
| `Embedding` | token id를 벡터로 변환 | `nn.Embedding` |
| `RNN` | 순환 신경망 | sequence 처리 |
| `LSTM` | 장단기 기억 모델 | 장기 의존성 완화 |
| `BiLSTM` | 양방향 LSTM | 앞뒤 문맥 사용 |
| `hn` | hidden state | LSTM 마지막 은닉 상태 |
| `cn` | cell state | LSTM 기억 상태 |
| `logits` | class 점수 | softmax 전 값 |
| `AdamW` | Adam + weight decay | Transformer 계열에서 자주 사용 |
| `DistilBERT` | 경량화된 BERT | 빠른 Transformer 분류 |
| `DataCollatorWithPadding` | 동적 padding | batch별 길이 맞춤 |
| `Trainer` | HF 학습 도구 | 학습 루프 자동화 |
| `precision` | 정밀도 | 예측한 양성 중 진짜 양성 비율 |
| `recall` | 재현율 | 실제 양성 중 맞힌 비율 |
| `f1` | precision/recall 조화 평균 | 불균형 데이터에서 중요 |

## 26. 시험용 요약

```text
17강 핵심 = 텍스트를 숫자 sequence로 바꾸고, BiLSTM 또는 DistilBERT로 감성 분류를 수행하는 것
```

꼭 기억할 것:

- NLP는 자연어를 컴퓨터가 계산 가능한 숫자로 바꾸는 과정이 중요하다.
- 이미지 분류는 image와 label이고, 자연어 분류는 text와 label이다.
- 토큰화는 문장을 작은 token 단위로 자르는 것이다.
- 어휘사전은 token과 숫자 id를 연결한다.
- `stoi`는 string to index다.
- `itos`는 index to string이다.
- `<pad>`는 길이를 맞추기 위한 token이다.
- `<unk>`는 사전에 없는 token을 처리하기 위한 token이다.
- padding은 짧은 문장을 같은 길이로 채우는 것이다.
- truncation은 긴 문장을 최대 길이에서 자르는 것이다.
- embedding은 token id를 dense vector로 바꾼다.
- LSTM은 sequence 순서를 처리하는 모델이다.
- BiLSTM은 문장을 앞뒤 양방향으로 읽는다.
- BiLSTM의 hidden은 forward와 backward를 `torch.cat(..., dim=1)`으로 연결한다.
- DistilBERT는 BERT를 경량화한 모델이다.
- BERT 계열은 Attention 기반으로 문맥 이해에 강하다.
- GPT 계열은 다음 token 생성에 강하다.
- `DataCollatorWithPadding`은 batch마다 동적 padding을 적용한다.
- `Trainer`는 Hugging Face 모델 학습 루프를 대신 처리한다.
- accuracy만 보지 말고 precision, recall, f1도 함께 본다.